In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("Load Data Bronze") \
    .config("spark.sql.shuffle.partitions", "200")  \
    .config("spark.sql.files.maxPartitionBytes", "128MB") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

In [0]:
%sql
GRANT USE CATALOG ON CATALOG vendas_pecas TO `joaopauloaraujopereira18@gmail.com`;
GRANT USE SCHEMA ON SCHEMA vendas_pecas.default TO `joaopauloaraujopereira18@gmail.com`;
GRANT CREATE TABLE ON SCHEMA vendas_pecas.default TO `joaopauloaraujopereira18@gmail.com`;
GRANT USE CATALOG ON CATALOG vendas_pecas TO `joaopauloaraujopereira18@gmail.com`;
GRANT ALL PRIVILEGES ON VOLUME vendas_pecas.default.my TO `joaopauloaraujopereira18@gmail.com`;
GRANT USE SCHEMA ON SCHEMA vendas_pecas.default TO `joaopauloaraujopereira18@gmail.com`;

In [0]:
path_in = "/Volumes/vendas_pecas/default/my/processar/vendas_pecas_automotiva.csv"
bronze_path = "/Volumes/vendas_pecas/default/my/bronze"

In [0]:
dbutils.fs.mkdirs("/Volumes/vendas_pecas/default/my/bronze")

In [0]:
dbutils.fs.mkdirs("/Volumes/vendas_pecas/default/my/processar")

In [0]:
schema_sales = StructType([
    StructField("id_nf", StringType(), True),
    StructField("data_venda", DateType(), True),
    StructField("cliente_id", LongType(), True),
    StructField("cliente_nome", StringType(), True),
    StructField("marca_carro", StringType(), True),
    StructField("modelo_carro", StringType(), True),
    StructField("produto_peca", StringType(), True),
    StructField("valor_unitario", DoubleType(), True),
    StructField("quantidade", DoubleType(), True),
    StructField("custo_unitario", DoubleType(), True),
    StructField("loja_id", IntegerType(), True),
    StructField("loja_nome", StringType(), True),
    StructField("grupo_loja", StringType(), True),
    StructField("vendedor_id", StringType(), True),
    StructField("vendedor_nome", StringType(), True)
])

# Leitura dos dados e adição da coluna nome do arquivo durante a leitura
df_vendas = spark.read.option("header", "true").schema(schema_sales).csv(path_in) 
                      



In [0]:
df_vendas.write.mode("overwrite").parquet(bronze_path)

In [0]:
df_vendas.write.format("delta").mode("overwrite").saveAsTable("vendas_pecas.camada_bronze.tb_bronze")

In [0]:
%fs ls '/Volumes/workspace/default/myvolume/bronze/'
